In [1]:
import requests

import geopandas as gpd

import pandas as pd

from sqlalchemy import create_engine

from dotenv import load_dotenv

from pathlib import Path

import os

In [2]:
# Load environment variables containing database credentials and data source URLs
env_path = Path("../.env")
load_dotenv(env_path)

# Retrieve connection parameters from the .env file
DATABASE_URL = os.getenv('DATABASE_URL')
RCN_URL = os.getenv('RCN_URL')

# Create a SQLAlchemy engine for database operations
engine = create_engine(DATABASE_URL)

### Download RCN Dataset

Download the official Real Estate Price Register (RCN) dataset for further processing and analysis.

In [3]:
r_rcn = requests.get(RCN_URL)

with open('rcn.parquet', 'wb') as f:
    f.write(r_rcn.content)

print(f'Status code: {r_rcn.status_code}')
print('RCN file downloaded successfully.')

Status code: 200
RCN file downloaded successfully.


### Load RCN Dataset

In [4]:
df_rcn = gpd.read_parquet('rcn.parquet')

print(f'Downloaded {len(df_rcn)} transactions.')

Downloaded 319065 transactions.


In [5]:
df_rcn.head()

,fid,gid,serwis_rcn,teryt,tran_przestrzen_nazw,tran_lokalny_id_iip,tran_wersja_id,tran_rodzaj_trans,tran_rodzaj_rynku,tran_sprzedajacy,...,lok_nr_lokalu,lok_funkcja,lok_liczba_izb,lok_nr_kond,lok_pow_uzyt,lok_pow_przyn,lok_cena_brutto,lok_vat,lok_adres,geometry
0,1,5953976,None,1465,PL.PZGiK.5346.RCN,968654f5-7a95-45d2-a1ef-a529aa4d6428,2018-10-30T15:17:54,wolnyRynek,wtorny,osobaFizyczna,...,18_LOK,mieszkalna,4,2.0,49.36,NaN,460000.00,NaN,MSC:Warszawa;UL:ulica Mordechaja Anielewicza;N...,POINT EMPTY
1,2,5961702,None,1465,PL.PZGiK.5346.RCN,4439eb7c-404f-4166-bae7-93df63f226be,2018-01-10T07:21:42,wolnyRynek,wtorny,osobaFizyczna,...,29_LOK,mieszkalna,4,3.0,58.47,NaN,600000.00,NaN,MSC:Warszawa;UL:Aleja Wojska Polskiego;NR_PORZ:31,POINT EMPTY
2,3,5962562,None,1465,PL.PZGiK.5346.RCN,3e082218-0ba1-47cd-b928-d203216e865e,2017-10-09T14:33:55,wolnyRynek,wtorny,osobaFizyczna,...,143_LOK,garaz,0,NaN,3234.62,NaN,25000.00,NaN,MSC:Warszawa;UL:ulica Wileńska;NR_PORZ:14,POINT EMPTY
3,4,5975986,None,1465,PL.PZGiK.5346.RCN,c0e231e8-fd7e-4d2d-8c27-f4b647c1e23d,2020-08-06T12:51:56,wolnyRynek,pierwotny,osobaPrawna,...,137_LOK,mieszkalna,4,7.0,67.48,NaN,798206.63,NaN,MSC:Warszawa;UL:ulica Fort Służew;NR_PORZ:3,POINT EMPTY
4,5,5989824,None,1465,PL.PZGiK.5346.RCN,a31bd3dc-d272-4ffc-9510-510febef7523,2020-11-27T07:59:44,wolnyRynek,pierwotny,osobaPrawna,...,201_LOK,mieszkalna,3,7.0,62.39,NaN,455447.00,NaN,MSC:Warszawa;UL:ulica Marcina Kasprzaka;NR_POR...,POINT EMPTY


In [6]:
df_rcn.columns

Index(['fid', 'gid', 'serwis_rcn', 'teryt', 'tran_przestrzen_nazw',
       'tran_lokalny_id_iip', 'tran_wersja_id', 'tran_rodzaj_trans',
       'tran_rodzaj_rynku', 'tran_sprzedajacy', 'tran_kupujacy',
       'tran_cena_brutto', 'tran_vat', 'dok_data', 'nier_rodzaj', 'nier_prawo',
       'nier_udzial', 'nier_pow_gruntu', 'nier_cena_brutto', 'nier_vat',
       'lok_id_lokalu', 'lok_nr_lokalu', 'lok_funkcja', 'lok_liczba_izb',
       'lok_nr_kond', 'lok_pow_uzyt', 'lok_pow_przyn', 'lok_cena_brutto',
       'lok_vat', 'lok_adres', 'geometry'],
      dtype='object')

In [7]:
# Convert transaction date to datetime format
df_rcn['transaction_date'] = pd.to_datetime(df_rcn['dok_data'])

### Filter Residential Apartment Transactions

Apply a set of business rules to retain only relevant apartment transactions for market analysis:

- `lok_funkcja == 'mieszkalna'` - Keep residential properties only.

- `transaction_date >= '2022-01-01'` - Limit the analysis to recent market activity.

- `lok_cena_brutto.notna()` - Remove transactions with missing apartment prices.

- `nier_udzial in ['1/1', '']` - Keep transactions involving full ownership and exclude partial ownership shares.

- `tran_rodzaj_trans == 'wolnyRynek'` - Retain only free-market transactions.

- `tran_sprzedajacy in ['osobaFizyczna', 'osobaPrawna']` - Keep transactions conducted by individuals or legal entities.

- `nier_prawo == 'wlasnoscLokaluWrazZPrawemZwiazanym'` - Retain apartments with full ownership rights and associated property rights.

In [8]:
df_warsaw = df_rcn[
    (df_rcn['lok_funkcja'] == 'mieszkalna')
    &
    (df_rcn['transaction_date'] >= '2022-01-01 00:00:00')
    &
    (df_rcn["lok_cena_brutto"].notna())
    &
    (df_rcn['nier_udzial'].isin(['1/1', '']))
    &
    (df_rcn['tran_rodzaj_trans'] == 'wolnyRynek')
    &
    (df_rcn['tran_sprzedajacy'].isin(['osobaFizyczna', 'osobaPrawna']))
    &
    (df_rcn['nier_prawo'] == 'wlasnoscLokaluWrazZPrawemZwiazanym')
].copy()

len_df_warsaw = len(df_warsaw)

print(f'Warsaw transactions: {len_df_warsaw}.')

Warsaw transactions: 93668.


### Prepare Analysis Dataset

Create additional features and standardize the dataset structure for analysis and visualization.

In [9]:
# Extract address components
df_warsaw["street"] = df_warsaw["lok_adres"].str.extract(r'UL:(.*?);NR_PORZ:')

df_warsaw["house_number"] = df_warsaw["lok_adres"].str.extract(r'NR_PORZ:(.*)$')

df_warsaw["nr_apartment"] = df_warsaw["lok_nr_lokalu"].str.replace("_LOK", "")



# Create analysis features
df_warsaw['price_per_m2'] = (df_warsaw['lok_cena_brutto'] / df_warsaw["lok_pow_uzyt"]).round(2)



# Convert coordinates to WGS84 and extract latitude/longitude
df_warsaw_geo = df_warsaw.to_crs(4326)

df_warsaw['longitude'] = df_warsaw_geo.geometry.x

df_warsaw['latitude'] = df_warsaw_geo.geometry.y



# Select and rename analysis fields
df_warsaw = df_warsaw[[
    'tran_lokalny_id_iip',
    'transaction_date',
    'tran_rodzaj_rynku',
    'nr_apartment',
    'street',
    'house_number',
    'lok_pow_uzyt',
    'lok_cena_brutto',
    'price_per_m2',
    'geometry',
    'longitude',
    'latitude'
]]

df_warsaw = df_warsaw.rename(columns={
    'tran_lokalny_id_iip': 'transaction_id',
    'tran_rodzaj_rynku': 'market_type',
    'lok_pow_uzyt': 'area',
    'lok_cena_brutto': 'price',
    'lok_adres': 'address'
})

### Explore Numerical Variables

Review the distribution of apartment area and price per square meter using descriptive statistics and selected percentiles.

The analysis helps identify potential outliers and supports the definition of data quality rules for further processing.

In [10]:
df_warsaw[['area', 'price_per_m2']].describe(
    percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99]
)

,area,price_per_m2
count,93668.000000,9.366800e+04
mean,56.351755,1.451818e+04
std,36.864826,6.568292e+03
min,0.850000,7.320000e+00
1%,22.626700,5.710155e+03
5%,27.970000,8.251282e+03
25%,38.700000,1.132065e+04
50%,49.410000,1.386760e+04
75%,65.560000,1.685914e+04
95%,107.240000,2.282874e+04


### Remove Extreme Values

Filter out unrealistic observations based on apartment area and price per square meter thresholds.

Applied rules:

- Area between 15 m² and 300 m²
- Price per m² between 5,000 PLN and 50,000 PLN

These thresholds remove extreme values and data quality issues while preserving the vast majority of market transactions.

In [11]:
df_warsaw = df_warsaw[
    (df_warsaw["area"] >= 15)
    &
    (df_warsaw["area"] <= 300)
    &
    (df_warsaw["price_per_m2"] >= 5000)
    &
    (df_warsaw["price_per_m2"] <= 50000)
].reset_index(drop=True)

len_df_warsaw_clear = len(df_warsaw)

print(f'Removed {len_df_warsaw - len_df_warsaw_clear} outliers.')
print(f'Warsaw transactions after cleaning: {len_df_warsaw_clear}.')

Removed 820 outliers.
Warsaw transactions after cleaning: 92848.


In [12]:
df_warsaw.head()

,transaction_id,transaction_date,market_type,nr_apartment,street,house_number,area,price,price_per_m2,geometry,longitude,latitude
0,2c0e1645-ec01-4a24-88fb-295cf5d8f6c9,2024-02-15 23:00:00+00:00,wtorny,41,ulica Ogrodowa,11,34.36,605000.00,17607.68,POINT (635857.889 487749.14),20.990287,52.238994
1,e746cd67-a678-47b9-935c-f0e481f281de,2024-05-14 22:00:00+00:00,wtorny,1,ulica 1 Sierpnia,35,37.80,540000.00,14285.71,POINT (634667.971 482603.65),20.970818,52.193039
2,4302bf29-3a29-425d-a9e3-489072f7d0de,2024-08-29 22:00:00+00:00,pierwotny,1,ulica Wylot,6,122.79,2296446.58,18702.23,POINT (631334.42 483120.339),20.922263,52.198487
3,7ef696c9-431c-43da-a2b2-ba51191d4897,2024-05-16 22:00:00+00:00,pierwotny,103,ulica Sarmacka,25,61.55,940000.00,15272.14,POINT (642549.632 478474.501),21.084347,52.153946
4,53eb2542-bfe4-48a0-818b-ac901052f52e,2025-05-28 22:00:00+00:00,wtorny,2,ulica Przylaszczkowa,19C,34.65,650000.00,18759.02,POINT (650165.913 478873.488),21.195801,52.155512


### Load District Boundaries

Load Warsaw district polygons from PostgreSQL for spatial analysis and district assignment.

In [13]:
df_warsaw_districts = gpd.read_postgis(
    'SELECT * FROM dim_districts',
    engine,
    geom_col='geometry'
)

print(f'Warsaw disricts: {len(df_warsaw_districts)}.')

Warsaw disricts: 18.


In [14]:
df_warsaw_districts.head()

,district,geometry
0,Żoliborz,"MULTIPOLYGON (((636711.52 490043.16, 636719.92..."
1,Śródmieście,"MULTIPOLYGON (((640176.94 486062.56, 640173.28..."
2,Ursus,"MULTIPOLYGON (((630540.86 482780.65, 630535.96..."
3,Bemowo,"MULTIPOLYGON (((630430.56 491372.53, 630310.06..."
4,Białołęka,"MULTIPOLYGON (((634919.44 493878.47, 634949.88..."


### Assign Districts to Transactions

Perform a spatial join to assign each apartment transaction to a Warsaw district based on its geographic location.

Transactions located outside district boundaries are flagged for further review.

In [15]:
df_warsaw = gpd.sjoin(
    df_warsaw,
    df_warsaw_districts,
    how="left",
    predicate="within"
)

df_warsaw = df_warsaw.drop(columns=['geometry', 'index_right'])

print(f"Transactions whithout district: {df_warsaw['district'].isna().sum()}.")

Transactions whithout district: 0.


In [16]:
df_warsaw.head()

,transaction_id,transaction_date,market_type,nr_apartment,street,house_number,area,price,price_per_m2,longitude,latitude,district
0,2c0e1645-ec01-4a24-88fb-295cf5d8f6c9,2024-02-15 23:00:00+00:00,wtorny,41,ulica Ogrodowa,11,34.36,605000.00,17607.68,20.990287,52.238994,Wola
1,e746cd67-a678-47b9-935c-f0e481f281de,2024-05-14 22:00:00+00:00,wtorny,1,ulica 1 Sierpnia,35,37.80,540000.00,14285.71,20.970818,52.193039,Włochy
2,4302bf29-3a29-425d-a9e3-489072f7d0de,2024-08-29 22:00:00+00:00,pierwotny,1,ulica Wylot,6,122.79,2296446.58,18702.23,20.922263,52.198487,Włochy
3,7ef696c9-431c-43da-a2b2-ba51191d4897,2024-05-16 22:00:00+00:00,pierwotny,103,ulica Sarmacka,25,61.55,940000.00,15272.14,21.084347,52.153946,Wilanów
4,53eb2542-bfe4-48a0-818b-ac901052f52e,2025-05-28 22:00:00+00:00,wtorny,2,ulica Przylaszczkowa,19C,34.65,650000.00,18759.02,21.195801,52.155512,Wawer


### Load Transactions to PostgreSQL

Store the processed apartment transaction dataset in PostgreSQL for dashboard development and analytical queries.

In [17]:
df_warsaw.to_sql(
    "fact_apartment_sales",
    engine,
    if_exists="replace",
    index=False
)

848